# Day 8 · Plotly EDA 2차: 최근 6년 변화와 챗봇용 통계 질문

**목표:** 2020~2025 산업중분류 × 사업장 규모 자료를 한 형태로 읽고, 집계 함수와 Plotly Figure 함수를 분리해 이후 Pandas/SQL 챗봇에서 재사용한다. 이 노트북은 로컬 CSV만 읽으며 **Run All**로 바로 실행된다.

공식 출처: [사고재해자수](https://www.data.go.kr/data/15084672/fileData.do), [사고사망자수](https://www.data.go.kr/data/15084674/fileData.do), [사망만인율](https://www.data.go.kr/data/15064491/fileData.do)의 주기성 과거 데이터. 새로 받은 2020~2024 CSV는 `data/history/`에 UTF-8로 저장했고 **기존 2025 원본 파일명·내용은 변경하지 않았다.**

**자료 공백:** 포털의 사망만인율 이력에는 2021년 파일이 없다. 2020년 사망만인율은 37개 산업중분류로, 다른 연도의 30개 분류와 일부 범위가 다르다. 사고 인원 자료도 2020~2021년에 두 업종의 표기가 2025년과 다르다. 없는 값이나 다른 분류를 임의로 채우지 않는다.


## 1. 파일 경로 매핑

`FILE_MAP[(지표, 연도)]`가 모든 로컬 파일의 단일 진입점이다. 2020~2024는 `data/history/`의 일관된 이름, 2025는 Day 7과 같은 **원본 이름**을 사용한다. `fatality_rate_2021.csv`는 원본이 없어 매핑에도 없다.


In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display

cwd = Path.cwd().resolve()
candidates = [cwd, *(parent / 'src/rag_project_1/Project_1' for parent in (cwd, *cwd.parents))]
PROJECT_DIR = next(p for p in candidates if p.name == 'Project_1' and (p / 'data').is_dir())
DATA_DIR = PROJECT_DIR / 'data'
YEARS = list(range(2020, 2026))
SIZE_ORDER = ['5인 미만', '5-9인', '10-19인', '20-29인', '30-49인',
              '50-99인', '100-299인', '300-499인', '500-999인', '1000인 이상']
METRIC_NAMES = {'accident_injured': '사고재해자수',
                'accident_death': '사고사망자수',
                'fatality_rate': '사망만인율'}
COUNT_METRICS = ('사고재해자수', '사고사망자수')

FILE_MAP = {(metric, year): DATA_DIR / 'history' / f'{slug}_{year}.csv'
            for slug, metric in METRIC_NAMES.items() for year in YEARS[:-1]
            if (DATA_DIR / 'history' / f'{slug}_{year}.csv').exists()}
FILE_MAP.update({(metric, 2025): next(DATA_DIR.glob(f'*산업중분류별 규모별 {metric}_20251231.csv'))
                 for metric in METRIC_NAMES.values()})

inventory = pd.DataFrame([
    {'연도': year, '지표': metric, '파일': str(FILE_MAP[(metric, year)].relative_to(PROJECT_DIR))
     if (metric, year) in FILE_MAP else '공식 파일 없음'}
    for year in YEARS for metric in METRIC_NAMES.values()
])
display(inventory)


## 2. 열 이름 표준화와 6년 데이터 통합

**열 정렬 각주:** 원본 CSV는 모두 `대업종`, `구분` 뒤에 규모 10개가 같은 순서로 있다. 다만 `5인~9인`/`5인-9인`/`5인 - 9인`, `1,000인`/`1000인`, 2024년 사고 건수 파일의 잘못된 `사업장수` 머리글처럼 **이름은 다르다**. Day 7처럼 열 **위치**를 공통 규모 순서에 대응시킨다. 산업 이름은 공백만 통일하며 서로 다른 분류를 강제로 합치지 않는다.


In [ ]:
# 각주 1: 원본의 규모 머리글 표기가 연도·지표마다 달라, 확인된 동일한 10열 순서로 정규화한다.
# 사망만인율의 공란은 산출 불가이므로 NaN으로 남긴다.
def load_stat_csv(metric: str, year: int, path: Path) -> pd.DataFrame:
    raw = pd.read_csv(path, encoding='utf-8-sig')
    if list(raw.columns[:2]) != ['대업종', '구분'] or len(raw.columns) != 12:
        raise ValueError(f'{path.name}: 산업중분류 × 규모 10열 파일이 아니다.')
    raw.columns = ['대업종', '산업중분류', *SIZE_ORDER]
    raw['산업중분류'] = raw['산업중분류'].astype(str).str.replace(r'\s+', '', regex=True)
    long = raw.melt(id_vars=['대업종', '산업중분류'], value_vars=SIZE_ORDER,
                    var_name='규모', value_name='값')
    long['값'] = pd.to_numeric(long['값'], errors='coerce')
    long['연도'] = year
    long['지표'] = metric
    return long[['연도', '대업종', '산업중분류', '규모', '지표', '값']]

statistics = pd.concat([load_stat_csv(metric, year, path)
                        for (metric, year), path in FILE_MAP.items()], ignore_index=True)
statistics['규모'] = pd.Categorical(statistics['규모'], categories=SIZE_ORDER, ordered=True)
statistics = statistics.sort_values(['연도', '지표', '산업중분류', '규모']).reset_index(drop=True)
display(statistics.head(10))
print('통합 행:', len(statistics), '| 연도:', sorted(statistics['연도'].unique()))


## 3. 연도별 구조·결측 확인

표의 `산업중분류 수`가 2020 사망만인율에서 더 크고 2021 사망만인율 행이 없는 것이 **원자료 상태**다. `2025와 동일명`은 연도별 업종 이름이 얼마나 연결되는지 보여준다. 범위가 다를 수 있는 업종은 이름만 비슷하다고 강제 병합하지 않는다.


In [ ]:
coverage = (statistics.groupby(['연도', '지표'], observed=True)
            .agg(산업중분류_수=('산업중분류', 'nunique'), 규모_수=('규모', 'nunique'),
                 행=('값', 'size'), 결측=('값', lambda s: s.isna().sum()))
            .reset_index())
# 각주: 2025년과 동일한 업종 이름의 수를 별도로 보여 분류 변경에 따른 추세 공백을 드러낸다.
reference_names = set(statistics.loc[statistics['연도'].eq(2025) &
                                     statistics['지표'].eq('사고사망자수'), '산업중분류'])
name_coverage = (statistics.groupby(['연도', '지표'])['산업중분류']
                 .apply(lambda names: len(set(names) & reference_names))
                 .reset_index(name='2025와 동일명'))
coverage = (pd.MultiIndex.from_product([YEARS, METRIC_NAMES.values()], names=['연도', '지표'])
            .to_frame(index=False).merge(coverage, on=['연도', '지표'], how='left')
            .merge(name_coverage, on=['연도', '지표'], how='left'))
display(coverage)
print('산업·규모·연도·지표 키 중복:',
      statistics.duplicated(['연도', '산업중분류', '규모', '지표']).sum())


## 4. 통계 질의 함수 — 그래프 코드와 분리

`query_stats`는 네 조건으로 원값을 찾는다. `sum_counts`는 **인원 수만** 합산한다. 사망만인율은 산업·규모별 비율을 단순 합산하거나 평균해 전체 비율로 만들지 않는다. `compare_sizes`, `six_year_trend`, `rate_benchmark`는 챗봇의 답변 재료를 돌려준다.


In [ ]:
# 각주 2: 네 조건을 선택적으로 적용해 원자료의 개별 산업·규모 값을 그대로 반환한다.
def query_stats(data, industry=None, size=None, year=None, metric=None):
    result = data
    for column, value in [('산업중분류', industry), ('규모', size), ('연도', year), ('지표', metric)]:
        if value is not None:
            result = result.loc[result[column].eq(value)]
    return result.sort_values(['연도', '산업중분류', '규모'])


# 각주 3: 사고 인원 수는 더할 수 있지만, 근로자 수 분모가 없는 사망만인율은 합산하지 않는다.
def sum_counts(data, metric, by=('연도',), **filters):
    if metric not in COUNT_METRICS:
        raise ValueError('합산은 사고재해자수·사고사망자수에만 사용하세요.')
    rows = query_stats(data, metric=metric, **filters)
    return rows.groupby(list(by), observed=True)['값'].sum(min_count=1).reset_index()


# 각주 4: 지정한 산업·연도에서 규모별 원값을 큰 순서로 정렬한다. 비율은 결측을 제외한다.
def compare_sizes(data, industry, year, metric):
    return (query_stats(data, industry=industry, year=year, metric=metric)
            .dropna(subset=['값'])[['규모', '값']].sort_values('값', ascending=False))


# 각주 5: 6개 연도를 강제로 나열하되 원본이 없는 연도는 NaN으로 남겨 선그래프가 연결되지 않게 한다.
def six_year_trend(data, industry, size, metric):
    rows = query_stats(data, industry=industry, size=size, metric=metric)
    return (rows.set_index('연도')['값'].reindex(YEARS).rename_axis('연도')
            .reset_index().rename(columns={'값': metric}))


# 각주 6: '높은 편'은 해당 산업의 다른 규모와, 같은 규모의 다른 산업 중앙값에 각각 비교한다.
# 중앙값은 비교 기준일 뿐 전체 근로자의 합산 사망만인율이 아니다.
def rate_benchmark(data, industry, size, year):
    rows = query_stats(data, year=year, metric='사망만인율')
    own = rows.loc[rows['산업중분류'].eq(industry) & rows['규모'].eq(size), '값']
    value = own.iloc[0] if not own.empty else float('nan')
    within = rows.loc[rows['산업중분류'].eq(industry) & rows['규모'].ne(size), '값'].median()
    across = rows.loc[rows['산업중분류'].ne(industry) & rows['규모'].eq(size), '값'].median()
    return pd.DataFrame([{'연도': year, '산업중분류': industry, '규모': size,
                          '해당 사망만인율': value, '같은 산업 다른 규모 중앙값': within,
                          '같은 규모 다른 산업 중앙값': across,
                          '같은 산업 내 높은 편': value > within if pd.notna(value) and pd.notna(within) else pd.NA}])


## 5. Plotly 함수 — 통계 질의 결과만 그리기

다음 두 함수는 통계를 다시 계산하지 않고 **집계 함수의 결과 DataFrame을 Figure로 바꾸기만** 한다. 이후 Streamlit에서는 반환된 Figure를 `st.plotly_chart()`에 넣을 수 있다.


In [ ]:
# 각주 7: compare_sizes()가 반환한 규모별 값만 표시하며 수치 집계는 수행하지 않는다.
def plot_size_bar(size_table, industry, year, metric):
    fig = px.bar(size_table.sort_values('값'), x='값', y='규모', orientation='h',
                 title=f'{year}년 {industry}: 규모별 {metric}', height=520)
    fig.update_traces(marker_color='#2563eb')
    return fig


# 각주 8: six_year_trend()의 2020~2025 행을 그대로 그린다. 결측 연도는 선으로 잇지 않는다.
def plot_six_year_line(trend_table, industry, size, metric):
    fig = px.line(trend_table, x='연도', y=metric, markers=True,
                  title=f'{industry} · {size}: {metric} (2020~2025)')
    fig.update_traces(connectgaps=False)
    fig.update_xaxes(tickmode='array', tickvals=YEARS)
    return fig


## 6. 챗봇 질문 ① — 이 업종에서 사고재해자가 가장 많은 규모는?

예시 업종은 `건설업`, 기준연도는 2025다. 다른 업종에도 같은 `compare_sizes()`를 적용할 수 있다.


In [ ]:
example_industry = '건설업'
example_size = '5인 미만'
latest_year = 2025
injured_by_size = compare_sizes(statistics, example_industry, latest_year, '사고재해자수')
display(injured_by_size)
plot_size_bar(injured_by_size, example_industry, latest_year, '사고재해자수').show()
first = injured_by_size.iloc[0]
display(Markdown(f"**답변 근거:** {latest_year}년 {example_industry}에서는 "
                 f"{first['규모']}이 {first['값']:,.0f}명으로 가장 많다."))


## 7. 챗봇 질문 ② — 이 산업의 소규모 사업장은 사망만인율이 높은 편인가?

`5인 미만`의 원값을 **같은 산업의 나머지 규모 중앙값** 및 **동규모의 다른 산업 중앙값**과 나란히 놓는다. 이는 상대적 비교이지 원인을 설명하거나 전체 사망만인율을 재계산한 값이 아니다.


In [ ]:
benchmark = rate_benchmark(statistics, example_industry, example_size, latest_year)
display(benchmark)
row = benchmark.iloc[0]
display(Markdown(
    f"**답변 근거:** {example_industry} · {example_size}의 {latest_year}년 사망만인율은 "
    f"{row['해당 사망만인율']:.2f}, 같은 산업 다른 규모 중앙값은 "
    f"{row['같은 산업 다른 규모 중앙값']:.2f}이다. "
    f"{'높은 편이다' if row['같은 산업 내 높은 편'] else '높은 편은 아니다'}."
))


## 8. 챗봇 질문 ③ — 이 산업·규모의 사고사망자수는 6년 동안 어떻게 변했나?

집계 표와 그래프를 따로 만든다. 사고 인원 파일은 6개 연도가 제공된다. 다만 2020~2021년 업종명이 달랐던 두 분류의 추세는 빈 구간으로 남는다. 특정 연도의 증감은 표의 실제 숫자로 말하고, 추세의 **원인은 추정하지 않는다**.


In [ ]:
death_trend = six_year_trend(statistics, example_industry, example_size, '사고사망자수')
display(death_trend)
plot_six_year_line(death_trend, example_industry, example_size, '사고사망자수').show()
start, end = death_trend['사고사망자수'].iloc[[0, -1]]
display(Markdown(f"**답변 근거:** {example_industry} · {example_size}의 사고사망자수는 "
                 f"2020년 {start:,.0f}명, 2025년 {end:,.0f}명이다. "
                 "연도별 값은 위 표에서 확인할 수 있다."))


### 사망만인율 6년 선그래프의 빈 구간

2021년 원본이 없으므로 선이 끊긴다. 일부 산업의 2020년 분류명도 이후와 다르므로 값이 없어 보일 수 있다. **보간하거나 0으로 대체하지 않는다.**


In [ ]:
rate_trend = six_year_trend(statistics, example_industry, example_size, '사망만인율')
display(rate_trend)
plot_six_year_line(rate_trend, example_industry, example_size, '사망만인율').show()


## 9. 챗봇 질문 ④ — 사망자수가 많은 곳과 사망만인율이 높은 곳은 왜 다른가?

순위 비교 예시는 `도소매·음식·숙박업`, 2025년이다. **사고사망자수는 절대 인원**, 포털 정의의 **사망만인율은 임금근로자 1만명당 사망자수**다. 서로 다른 지표여서 순위가 다를 수 있지만, 제공된 CSV에는 분모 근로자수와 분자 범위의 검증 정보가 없어 특정 원인까지 단정할 수 없다.


In [ ]:
comparison_industry = '도소매·음식·숙박업'
death_sizes = compare_sizes(statistics, comparison_industry, latest_year, '사고사망자수')
rate_sizes = compare_sizes(statistics, comparison_industry, latest_year, '사망만인율')
display(pd.DataFrame({'사고사망자수 상위 규모': death_sizes['규모'].head(5).to_numpy(),
                      '사고사망자수': death_sizes['값'].head(5).to_numpy(),
                      '사망만인율 상위 규모': rate_sizes['규모'].head(5).to_numpy(),
                      '사망만인율': rate_sizes['값'].head(5).to_numpy()}))
display(Markdown(
    f"**답변 근거:** {latest_year}년 {comparison_industry}에서 사고사망자수 1위는 "
    f"{death_sizes.iloc[0]['규모']}({death_sizes.iloc[0]['값']:,.0f}명), "
    f"사망만인율 1위는 {rate_sizes.iloc[0]['규모']}({rate_sizes.iloc[0]['값']:.2f})이다. "
    "절대 인원과 근로자수 기준 비율의 순위 차이까지 설명할 수 있으나, "
    "이 자료만으로 현장별 원인을 특정할 수 없다."
))


## 10. 이후 챗봇이 답할 수 있는 통계 질문

| 질문 형태 | 사용할 데이터·함수 | 답변 시 표시할 근거 |
| --- | --- | --- |
| “이 업종에서 사고재해자가 가장 많은 규모는?” | `compare_sizes(statistics, 업종, 연도, '사고재해자수')` | 규모별 인원과 기준연도 |
| “이 산업의 소규모 사업장은 사망만인율이 높은 편이야?” | `rate_benchmark(statistics, 업종, 규모, 연도)` | 해당 값, 같은 산업 다른 규모 중앙값, 동규모 다른 산업 중앙값 |
| “이 산업·규모의 사고사망자수는 최근 6년 어떻게 변했어?” | `six_year_trend(statistics, 업종, 규모, '사고사망자수')` | 2020~2025 실제 값과 결측 여부 |
| “사망자수와 사망만인율 순위는 왜 달라?” | 두 지표의 `compare_sizes` 결과 | 절대 인원과 근로자 1만명당 비율의 차이, 원인 추정 금지 |

`query_stats`는 `산업중분류 / 규모 / 연도 / 지표`로 원값을 조회하고, `sum_counts`는 **사고 인원 수만** 필요한 단위로 합산한다. 사망만인율의 2021년 공백과 2020년 분류 차이를 숨기지 않으면 이 구조를 Pandas/SQL 챗봇에도 옮길 수 있다. 예측·Streamlit·RAG 연결은 여기서 다루지 않는다.
